<a href="https://colab.research.google.com/github/venkatasnehith/-Fraud-transaction-receipt-and-loan-complaint-detection.-/blob/main/Fraud_transaction_receipt_and_loan_complaint_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn tensorflow keras opencv-python pytesseract --quiet


In [2]:
from google.colab import files
uploaded = files.upload()  # Select your dataset file (CSV, Excel, etc.)


Saving archive (6).zip to archive (6).zip


In [6]:
import zipfile
import pandas as pd

# Unzip the file
with zipfile.ZipFile("archive (6).zip", 'r') as zip_ref:
    zip_ref.extractall("unzipped_data")
    print("📁 Files inside zip:")
    print(zip_ref.namelist())  # Lists all files in the zip

# Load a specific CSV after viewing the list
# Load a specific CSV file (e.g., loan_applications.csv)
df = pd.read_csv("unzipped_data/loan_applications.csv")
df.head()



📁 Files inside zip:
['loan_applications.csv', 'transactions.csv']


,application_id,customer_id,application_date,loan_type,loan_amount_requested,loan_tenure_months,interest_rate_offered,purpose_of_loan,employment_status,monthly_income,...,existing_emis_monthly,debt_to_income_ratio,property_ownership_status,residential_address,applicant_age,gender,number_of_dependents,loan_status,fraud_flag,fraud_type
0,c8bf0bea-70e6-4870-9125-41b8210c527f,CUST109427,2023-04-09,Business Loan,604000.0,12,11.66,Medical Emergency,Retired,34700.0,...,1100.0,3.17,Rented,"94/31, Sehgal Zila, Vadodara-380521, Anantapur...",28,Female,3,Approved,0,NaN
1,91224cec-3544-4bc7-ac15-a9792da54c02,CUST106146,2023-09-23,Car Loan,100000.0,240,13.62,Education,Unemployed,51600.0,...,0.0,0.00,Owned,"H.No. 00, Sheth Chowk, Ichalkaranji 006728, Im...",44,Other,3,Approved,0,NaN
2,4efcd02d-4a03-4ab7-9bd1-0ff430493d0c,CUST100674,2023-05-22,Education Loan,431000.0,60,11.40,Medical Emergency,Self-Employed,14800.0,...,4600.0,31.08,Rented,"H.No. 81, Dutta Path, Kozhikode-340301, Tadepa...",56,Other,4,Approved,0,NaN
3,a61337d4-ba04-4a68-b492-2cb8266e6ed7,CUST106466,2024-07-09,Car Loan,324000.0,120,10.36,Debt Consolidation,Self-Employed,28800.0,...,4000.0,13.89,Rented,"H.No. 022, Rege Road, Tiruvottiyur-927857, Aur...",27,Other,4,Declined,0,NaN
4,a8d1639e-170b-41b2-826a-55c7dae38d16,CUST112319,2023-11-20,Personal Loan,100000.0,36,14.14,Business Expansion,Salaried,43900.0,...,1100.0,2.51,Rented,"85/24, Bali Zila, Sambalpur 922071, Tumkur, Ke...",50,Other,0,Declined,0,NaN


In [12]:
print(df.columns)
df.head()


Index(['application_id', 'customer_id', 'application_date', 'loan_type',
       'loan_amount_requested', 'loan_tenure_months', 'interest_rate_offered',
       'purpose_of_loan', 'employment_status', 'monthly_income', 'cibil_score',
       'existing_emis_monthly', 'debt_to_income_ratio',
       'property_ownership_status', 'residential_address', 'applicant_age',
       'gender', 'number_of_dependents', 'loan_status', 'fraud_flag',
       'fraud_type'],
      dtype='object')


,application_id,customer_id,application_date,loan_type,loan_amount_requested,loan_tenure_months,interest_rate_offered,purpose_of_loan,employment_status,monthly_income,...,existing_emis_monthly,debt_to_income_ratio,property_ownership_status,residential_address,applicant_age,gender,number_of_dependents,loan_status,fraud_flag,fraud_type
0,c8bf0bea-70e6-4870-9125-41b8210c527f,CUST109427,2023-04-09,Business Loan,604000.0,12,11.66,Medical Emergency,Retired,34700.0,...,1100.0,3.17,Rented,"94/31, Sehgal Zila, Vadodara-380521, Anantapur...",28,Female,3,Approved,0,NaN
1,91224cec-3544-4bc7-ac15-a9792da54c02,CUST106146,2023-09-23,Car Loan,100000.0,240,13.62,Education,Unemployed,51600.0,...,0.0,0.00,Owned,"H.No. 00, Sheth Chowk, Ichalkaranji 006728, Im...",44,Other,3,Approved,0,NaN
2,4efcd02d-4a03-4ab7-9bd1-0ff430493d0c,CUST100674,2023-05-22,Education Loan,431000.0,60,11.40,Medical Emergency,Self-Employed,14800.0,...,4600.0,31.08,Rented,"H.No. 81, Dutta Path, Kozhikode-340301, Tadepa...",56,Other,4,Approved,0,NaN
3,a61337d4-ba04-4a68-b492-2cb8266e6ed7,CUST106466,2024-07-09,Car Loan,324000.0,120,10.36,Debt Consolidation,Self-Employed,28800.0,...,4000.0,13.89,Rented,"H.No. 022, Rege Road, Tiruvottiyur-927857, Aur...",27,Other,4,Declined,0,NaN
4,a8d1639e-170b-41b2-826a-55c7dae38d16,CUST112319,2023-11-20,Personal Loan,100000.0,36,14.14,Business Expansion,Salaried,43900.0,...,1100.0,2.51,Rented,"85/24, Bali Zila, Sambalpur 922071, Tumkur, Ke...",50,Other,0,Declined,0,NaN


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop columns that won't help in prediction (IDs, address, etc.)
X = df.drop(columns=['application_id', 'customer_id', 'residential_address', 'fraud_flag', 'fraud_type'])
y = df['fraud_flag']  # Target column

# One-hot encode categorical columns
X = pd.get_dummies(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Optional: scale numeric features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9813
           1       1.00      1.00      1.00       187

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("✅ Model trained!")


✅ Model trained!


Fraud Transaction Receipt Detection

In [29]:
import cv2
import pytesseract
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()

for fn in uploaded.keys():
    img = Image.open(io.BytesIO(uploaded[fn]))
    text = pytesseract.image_to_string(img)
    print("Extracted Text:\n", text)

    # Convert text to model input (example logic)
    # processed_input = process_text_to_model_format(text)
    # prediction = model.predict([processed_input])
    # print("🚨 Prediction:", "FRAUD" if prediction[0] == 1 else "NOT FRAUD")


Saving Screenshot 2025-07-31 223602.png to Screenshot 2025-07-31 223602.png
Extracted Text:
 LAST MONTH RENT

MONTH 05 FOR LANDLORD
MON, MAY 01, 2023

 

QTY ITEM AMT
01 INTERBANK GIRO ABUSE -RMXXX
02 COMPANY INVOICES ~RMXXX

 

BALANCE: -RMXXXX

   

STATUS: PENDING



In [30]:
def process_text_to_model_format(text):
    # Dummy feature vector with same number of features as model was trained on
    # Replace this with real logic to parse features like amount, status, name, etc.
    return [0]*X.shape[1]  # X = your training feature set


In [31]:
# Run OCR
img = Image.open(io.BytesIO(uploaded[fn]))
text = pytesseract.image_to_string(img)
print("Extracted Text:\n", text)

# Dummy conversion to model input
processed_input = process_text_to_model_format(text)

# Run model prediction
prediction = model.predict([processed_input])
print("🚨 Prediction:", "FRAUD" if prediction[0] == 1 else "NOT FRAUD")


Extracted Text:
 LAST MONTH RENT

MONTH 05 FOR LANDLORD
MON, MAY 01, 2023

 

QTY ITEM AMT
01 INTERBANK GIRO ABUSE -RMXXX
02 COMPANY INVOICES ~RMXXX

 

BALANCE: -RMXXXX

   

STATUS: PENDING

🚨 Prediction: NOT FRAUD


In [48]:
# STEP 1: Install & import dependencies
!pip install pytesseract pillow scikit-learn --quiet

import pytesseract
from PIL import Image
from google.colab import files
import io
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# STEP 2: Define feature extraction logic
def process_text_to_model_format(text):
    text = text.lower()

    features = {
        "has_pending_status": int("pending" in text),
        "has_unknown_terms": int("unknown" in text or "abuse" in text or "adjusted" in text),
        "contains_gibberish": int("rmxxx" in text or "rx" in text),
        "mentions_invoice": int("invoice" in text),
        "mentions_balance_due": int("balance" in text and "due" in text),
    }

    return list(features.values())

# STEP 3: Create synthetic training data
training_data = [
    {"text": "Paid full invoice, balance cleared", "fraud": 0},
    {"text": "Transaction adjusted manually. Status: Pending", "fraud": 1},
    {"text": "INTERBANK GIRO ABUSE -RMXXX\nStatus: Pending", "fraud": 1},
    {"text": "Success. EMI paid ₹5,000 on time", "fraud": 0},
    {"text": "Status: Unknown. Invoice Missing. Balance Due", "fraud": 1},
]

df_train = pd.DataFrame(training_data)
X_train = df_train["text"].apply(process_text_to_model_format).tolist()
y_train = df_train["fraud"]

# STEP 4: Train a basic model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# STEP 5: Upload and test your receipt
uploaded = files.upload()

for fn in uploaded.keys():
    img = Image.open(io.BytesIO(uploaded[fn]))
    text = pytesseract.image_to_string(img)
    print("📄 Extracted Text:\n", text)

    processed_input = process_text_to_model_format(text)
    prediction = model.predict([processed_input])
    print("🔍 Prediction:", "🚨 FRAUD" if prediction[0] == 1 else "✅ NOT FRAUD")


Saving Screenshot 2025-07-31 223030.png to Screenshot 2025-07-31 223030 (1).png
📄 Extracted Text:
 Bistro Box Departures

TAX INVOICE

Take Away Quick Sale
Till Cashier
Invoice # 177957
Salesperson Ayla M

12:26 PM 3 Jul 24

FLAT WHITE-SMALL 5.80
CARROT CAKE 7.50
HAM & CHEESE TOASTIE 10.25
MUFFIN 5.00
BALANCE DUE $28.50
Includes GST

TENDERED $28.50

GST # 136-563-378
Ph: 021749899
Printed by onetap.systems

🔍 Prediction: ✅ NOT FRAUD


detect loan Complaints

In [47]:
# Install dependencies
!pip install scikit-learn pytesseract pillow --quiet

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import make_pipeline
from google.colab import files
from PIL import Image
import pytesseract
import io

# STEP 1: Dummy data for fraud detection
data = [
    {"text": "I applied for a loan and received proper confirmation.", "fraud": 0},
    {"text": "The agent asked me to pay ₹5000 upfront through UPI.", "fraud": 1},
    {"text": "I got a sanctioned letter with fake signatures.", "fraud": 1},
    {"text": "Loan approved and EMI schedule received.", "fraud": 0},
    {"text": "I got a message asking to click a suspicious link.", "fraud": 1},
    {"text": "Everything was processed through official app.", "fraud": 0},
]

df = pd.DataFrame(data)

# STEP 2: Train model
model = make_pipeline(CountVectorizer(), RandomForestClassifier())
model.fit(df['text'], df['fraud'])

# STEP 3: Upload any file (text or image)
uploaded = files.upload()

# STEP 4: Handle and predict
for fn in uploaded.keys():
    if fn.lower().endswith(('.png', '.jpg', '.jpeg')):
        # It's an image – extract text
        img = Image.open(io.BytesIO(uploaded[fn]))
        input_text = pytesseract.image_to_string(img)
        print(f"📸 Extracted Text from {fn}:\n", input_text)
    elif fn.lower().endswith('.txt'):
        # It's a text file
        with open(fn, 'r', encoding='utf-8', errors='ignore') as f:
            input_text = f.read()
        print(f"📄 Uploaded Complaint Text:\n", input_text)
    else:
        print(f"❌ Unsupported file type: {fn}")
        continue

    # Run prediction
    prediction = model.predict([input_text])
    print("🔍 Prediction:", "🚨 FRAUD" if prediction[0] == 1 else "✅ NOT FRAUD")


Saving Screenshot 2025-07-31 224738.png to Screenshot 2025-07-31 224738 (3).png
📸 Extracted Text from Screenshot 2025-07-31 224738 (3).png:
 The lawyer, Mr. Ashok Narwat,

has been appointed to file an
authorization/non-performance
complaint case against you in
Gurgaon, Haryana. This may result in
two years’ imprisonment or double
fines or both because you fail to
clear your dues. RupeeFund loan
https://wa.me/919564062672 for
settlement/payment or visit the
nearest Airtel store to pay in cash.
Pay online through the RupeeFund

app https://bitly/3s4vdVT gaa pny

Hello sir pls check | am not purchase

loan this app pls check 6:26 PMV

© Message Sea o

🔍 Prediction: 🚨 FRAUD
